# SAE-XCrash — NB07: Reduced 10-Feature Transfer Experiment
**Work Package WP7** | P13: Honest Zero-Shot Transfer (10 common features only)

Load-and-continue notebook. Run **after** NB01, NB02, NB05.
Addresses Reviewer 4 Comment 5: trains a reduced US XGBoost on the 10 informative
common features only, then applies zero-shot to STATS19 — eliminating the 12
mean-filled noise features that depress performance in the original zero-shot.

**SAFE:** This notebook is READ-ONLY for all existing files.
It writes ONLY to `logs/wp7/` and `figures/wp7/` — new directories.
No existing model, calibrator, parquet, or log file is modified.

### Steps
| Step | Description | Paper location |
|------|-------------|----------------|
| 1 | Environment & Paths | — |
| 2 | Load feature lists + HP log (read-only) | — |
| 3 | Load US train/val + STATS19 test parquets (read-only) | — |
| 4 | P13: Retrain XGBoost on 10 common features (US data) | Table 7 new row |
| 5 | Zero-shot transfer to STATS19 test | Table 7 new row |
| 6 | Compare vs original zero-shot + save results | logs/wp7/p13_reduced_transfer.json |

**GPU:** T4 sufficient. Reduced feature set trains much faster than full model.  
**Runtime:** ~15–20 min total on T4.  
**Seed:** `SEED = 42`

---
## Step 1 — Environment & Paths

In [1]:
!pip install -q xgboost scikit-learn pyarrow numpy pandas

import os, json, time, warnings, pickle
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Paths (read-only sources — matching NB02/NB05 conventions) ────────────────
PROJ_ROOT = Path('/content/drive/MyDrive/SAE-XCrash')
PROC_DIR = PROJ_ROOT / 'data' / 'processed'
LOGS_DIR_SRC = PROJ_ROOT / 'logs'                    # read-only source

# ── New output directories (WP7 only — nothing else is written) ───────────────
LOGS_DIR = PROJ_ROOT / 'logs'    / 'wp7'             # NEW — won't overwrite anything
FIGS_DIR = PROJ_ROOT / 'figures' / 'wp7'             # NEW
LOGS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
ECE_BINS = 15
DPI = 300

print(f'PROJ_ROOT : {PROJ_ROOT}')
print(f'Output    : {LOGS_DIR}')
print(f'NOTE: All source files are read-only. Only logs/wp7/ and figures/wp7/ are written.')

Mounted at /content/drive
PROJ_ROOT : /content/drive/MyDrive/SAE-XCrash
Output    : /content/drive/MyDrive/SAE-XCrash/logs/wp7
NOTE: All source files are read-only. Only logs/wp7/ and figures/wp7/ are written.


---
## Step 2 — Load Feature Lists + HP Log

In [2]:
# ── ECE helper (matching NB03 implementation) ─────────────────────────────────
def ece_score(y_true, y_prob, n_bins=ECE_BINS):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        ece += mask.sum() / n * abs(y_true[mask].mean() - y_prob[mask].mean())
    return round(float(ece), 6)

# ── Load feature lists from split_metadata.json (saved by NB01) ───────────────
print('Loading feature lists from split_metadata.json ...')
with open(PROC_DIR / 'split_metadata.json') as f:
    meta = json.load(f)

FEAT_USA = meta['usa']['feat_cols']
FEAT_S19 = meta['s19']['feat_cols']

# Derive FEAT_COMMON (10 informative features) — matching NB05 exactly
s19_set = set(FEAT_S19)
FEAT_COMMON = [f for f in FEAT_USA if f in s19_set]

print(f'  FEAT_USA    : {len(FEAT_USA)} features')
print(f'  FEAT_S19    : {len(FEAT_S19)} features')
print(f'  FEAT_COMMON : {len(FEAT_COMMON)} features')
print(f'  Common list : {FEAT_COMMON}')

# ── Load WP2 metadata (SPW) ───────────────────────────────────────────────────
with open(LOGS_DIR_SRC / 'wp2_meta.json') as f:
    wp2_meta = json.load(f)
SPW = float(wp2_meta.get('spw', 36.0))
print(f'\n  SPW (class weight): {SPW:.1f}')

# ── Load HP log for XGBoost best params ───────────────────────────────────────
print('\nLoading hyperparameter log ...')
hp_path = LOGS_DIR_SRC / 'wp2_hyperparameter_log.json'
if hp_path.exists():
    with open(hp_path) as f:
        hp_log = json.load(f)
    xgb_best = hp_log.get('xgb_best', {})
    print(f'  Loaded XGB best params: {list(xgb_best.keys())}')
else:
    print('  HP log not found — using paper Table B1 values as fallback')
    xgb_best = {
        'learning_rate': 0.0632, 'n_estimators': 1100, 'max_depth': 10,
        'subsample': 0.9448, 'colsample_bytree': 0.7279,
        'reg_alpha': 5.6e-6, 'reg_lambda': 1.3e-7,
    }

print('\nFeature lists and HP log loaded ✓')

Loading feature lists from split_metadata.json ...
  FEAT_USA    : 32 features
  FEAT_S19    : 22 features
  FEAT_COMMON : 10 features
  Common list : ['t_year', 't_month', 't_hour', 't_dayofweek', 't_is_weekend', 't_season', 't_is_rush', 't_is_night', 'w_weather_cat', 'w_is_dark']

  SPW (class weight): 36.0

Loading hyperparameter log ...
  Loaded XGB best params: ['max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'min_child_weight', 'reg_alpha', 'reg_lambda', 'gamma', 'n_estimators']

Feature lists and HP log loaded ✓


---
## Step 3 — Load Parquets (Read-Only)

In [3]:
# ── Load US train + val (for model training + calibration) ────────────────────
print('Loading US-Accidents parquets ...')
t0 = time.time()
usa_train = pd.read_parquet(PROC_DIR / 'usa_train_processed.parquet')
usa_val = pd.read_parquet(PROC_DIR / 'usa_val_processed.parquet')
print(f'  US Train: {len(usa_train):,} rows  ({time.time()-t0:.0f}s)')

y_us_train = usa_train['label'].values
y_us_val = usa_val['label'].values

# 10 common features only — no mean-filling needed
X_us_train_red = usa_train[FEAT_COMMON].values.astype(np.float32)
X_us_val_red = usa_val[FEAT_COMMON].values.astype(np.float32)

print(f'  US Val  : {len(usa_val):,} rows')
print(f'  No-skill AUPRC (US train): {y_us_train.mean():.4f}')

# Free full feature arrays — only need common features for this experiment
del usa_train, usa_val
import gc; gc.collect()
print('  Full US arrays freed from memory ✓')

# ── Load STATS19 test (for zero-shot evaluation) ───────────────────────────────
print('\nLoading STATS19 parquets ...')
t0 = time.time()
s19_val = pd.read_parquet(PROC_DIR / 's19_val_processed.parquet')
s19_test = pd.read_parquet(PROC_DIR / 's19_test_processed.parquet')
print(f'  S19 Val : {len(s19_val):,} rows')
print(f'  S19 Test: {len(s19_test):,} rows  ({time.time()-t0:.0f}s)')

y_s19_val = s19_val['label'].values
y_s19_test = s19_test['label'].values

# Extract common features from STATS19 (directly available — no mean-filling)
X_s19_val_red = s19_val[FEAT_COMMON].values.astype(np.float32)
X_s19_test_red = s19_test[FEAT_COMMON].values.astype(np.float32)

no_skill_s19 = float(y_s19_test.mean())
print(f'  No-skill AUPRC (S19 test): {no_skill_s19:.4f}')
print('\nAll parquets loaded ✓')

Loading US-Accidents parquets ...
  US Train: 5,549,870 rows  (17s)
  US Val  : 1,268,806 rows
  No-skill AUPRC (US train): 0.0271
  Full US arrays freed from memory ✓

Loading STATS19 parquets ...
  S19 Val : 106,004 rows
  S19 Test: 205,185 rows  (4s)
  No-skill AUPRC (S19 test): 0.2438

All parquets loaded ✓


---
## Step 4 — P13: Retrain XGBoost on 10 Common Features (US Data)

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# P13 — Reduced US Model
# Trains XGBoost using ONLY the 10 informative common features on US data.
# Same HP as the original US model (from wp2_hyperparameter_log.json).
# Same SPW and temporal split protocol.
# This eliminates the 12 mean-filled noise features that depress the
# original zero-shot AUROC from its true value (Reviewer 4, Comment 5).
# ─────────────────────────────────────────────────────────────────────────────
print('P13 — Retrain XGBoost on 10 common features (US data)')
print('=' * 60)
print(f'  Features used: {FEAT_COMMON}')

# ── XGBoost DMatrix ───────────────────────────────────────────────────────────
dtrain_red = xgb.DMatrix(X_us_train_red, label=y_us_train,
                         feature_names=FEAT_COMMON)
dval_red = xgb.DMatrix(X_us_val_red,   label=y_us_val,
                         feature_names=FEAT_COMMON)

# ── Parameters (same as original model; adapted for reduced feature set) ──────
params_red = {
    'learning_rate'    : xgb_best.get('learning_rate', 0.063),
    'max_depth'        : xgb_best.get('max_depth', 10),
    'subsample'        : xgb_best.get('subsample', 0.945),
    'colsample_bytree' : min(xgb_best.get('colsample_bytree', 0.728), 1.0),
    'reg_alpha'        : xgb_best.get('reg_alpha', 5.6e-6),
    'reg_lambda'       : xgb_best.get('reg_lambda', 1.3e-7),
    'scale_pos_weight' : SPW,
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'aucpr',
    'tree_method'      : 'hist',
    'device'           : 'cuda',
    'seed'             : SEED,
}

n_rounds = int(xgb_best.get('n_estimators', 1100))
print(f'\n  Training ({n_rounds} rounds max, early stop 50) ...')
t0 = time.time()

xgb_red = xgb.train(
    params_red,
    dtrain_red,
    num_boost_round = n_rounds,
    evals = [(dval_red, 'val')],
    early_stopping_rounds = 50,
    verbose_eval = 100,
)

train_time = time.time() - t0
print(f'\n  Best iteration : {xgb_red.best_iteration}')
print(f'  Training time  : {train_time:.0f}s')

# ── US val scores (for calibration) ──────────────────────────────────────────
p_red_us_val = xgb_red.predict(dval_red)
print(f'\n  US val AUPRC (reduced model): '
      f'{average_precision_score(y_us_val, p_red_us_val):.4f}')
print('\nP13 training complete ✓')

P13 — Retrain XGBoost on 10 common features (US data)
  Features used: ['t_year', 't_month', 't_hour', 't_dayofweek', 't_is_weekend', 't_season', 't_is_rush', 't_is_night', 'w_weather_cat', 'w_is_dark']

  Training (1317 rounds max, early stop 50) ...
[0]	val-aucpr:0.03307
[64]	val-aucpr:0.03407

  Best iteration : 14
  Training time  : 3s

  US val AUPRC (reduced model): 0.0341

P13 training complete ✓


---
## Step 5 — Zero-Shot Transfer to STATS19 Test

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Zero-shot: Apply reduced US model directly to STATS19 test.
# No mean-filling needed — STATS19 has all 10 common features natively.
# Calibration: fit isotonic on STATS19 val (matching NB05 protocol).
# ─────────────────────────────────────────────────────────────────────────────
print('Step 5 — Zero-shot transfer to STATS19 test')
print('=' * 60)

ds19_val_red = xgb.DMatrix(X_s19_val_red,  feature_names=FEAT_COMMON)
ds19_test_red = xgb.DMatrix(X_s19_test_red, feature_names=FEAT_COMMON)

# ── Zero-shot scores ──────────────────────────────────────────────────────────
p_red_s19_val = xgb_red.predict(ds19_val_red)
p_red_s19_test = xgb_red.predict(ds19_test_red)

# ── Calibration on STATS19 val (same protocol as NB05) ────────────────────────
iso_red = IsotonicRegression(out_of_bounds='clip')
iso_red.fit(p_red_s19_val, y_s19_val)
p_red_s19_test_cal = np.clip(iso_red.predict(p_red_s19_test), 0, 1)

# ── Metrics ───────────────────────────────────────────────────────────────────
reduced_results = {
    'model'      : 'XGBoost — Reduced zero-shot (10 common features)',
    'n_features' : len(FEAT_COMMON),
    'features'   : FEAT_COMMON,
    'AUROC'      : round(roc_auc_score(y_s19_test, p_red_s19_test), 4),
    'AUPRC'      : round(average_precision_score(y_s19_test, p_red_s19_test), 4),
    'ECE_uncal'  : ece_score(y_s19_test, p_red_s19_test),
    'ECE_cal'    : ece_score(y_s19_test, p_red_s19_test_cal),
    'Brier_cal'  : round(brier_score_loss(y_s19_test, p_red_s19_test_cal), 6),
    'train_time' : f'{train_time:.0f}s',
}

print('\n  Results:')
for k, v in reduced_results.items():
    if k not in ('features',):
        print(f'    {k:<15}: {v}')
print('\nZero-shot transfer complete ✓')

Step 5 — Zero-shot transfer to STATS19 test

  Results:
    model          : XGBoost — Reduced zero-shot (10 common features)
    n_features     : 10
    AUROC          : 0.5216
    AUPRC          : 0.2586
    ECE_uncal      : 0.197226
    ECE_cal        : 0.008757
    Brier_cal      : 0.184211
    train_time     : 3s

Zero-shot transfer complete ✓


---
## Step 6 — Comparison vs Original Zero-Shot + Save Results

In [6]:
# ── Reference values from paper Table 7 (NB05 results) ───────────────────────
original_zeroshot = {
    'model'  : 'XGBoost — Original zero-shot (22 features, 12 mean-filled)',
    'AUROC'  : 0.5182,
    'AUPRC'  : 0.2562,
    'ECE_uncal': 0.0688,
    'ECE_cal': 0.0088,
    'Brier_cal': 0.1843,
}

print('=' * 80)
print('COMPARISON — Table 7 (Cross-Dataset Transfer, STATS19 test year 2022)')
print('=' * 80)
print(f"{'Configuration':<52} {'AUROC':>8} {'AUPRC':>8} {'ECE(cal)':>10} {'Brier':>8}")
print('-' * 80)

# Original zero-shot (reference)
r = original_zeroshot
print(f"  {r['model']:<50} {r['AUROC']:>8.4f} {r['AUPRC']:>8.4f} "
      f"{r['ECE_cal']:>10.4f} {r['Brier_cal']:>8.4f}")

# New reduced zero-shot
r = reduced_results
print(f"  {r['model']:<50} {r['AUROC']:>8.4f} {r['AUPRC']:>8.4f} "
      f"{r['ECE_cal']:>10.4f} {r['Brier_cal']:>8.4f}  <- NEW")

# Delta
delta_auroc = reduced_results['AUROC'] - original_zeroshot['AUROC']
delta_auprc = reduced_results['AUPRC'] - original_zeroshot['AUPRC']
print(f"\n  Delta (reduced vs original zero-shot):")
print(f"    AUROC: {delta_auroc:+.4f}")
print(f"    AUPRC: {delta_auprc:+.4f}")
print(f"\n  No-skill AUPRC baseline (STATS19): {no_skill_s19:.4f}")

# ── Save results ──────────────────────────────────────────────────────────────
results = {
    'reduced_zero_shot'   : reduced_results,
    'original_zero_shot'  : original_zeroshot,
    'delta_auroc'         : round(delta_auroc, 4),
    'delta_auprc'         : round(delta_auprc, 4),
    'no_skill_auprc_s19'  : no_skill_s19,
    'note'                : (
        'Reduced model trained on 10 common features only (US data). '
        'No mean-filling needed for STATS19 — all 10 features available natively. '
        'Delta shows the performance cost of mean-filling 12 non-common features '
        'in the original zero-shot experiment.'
    ),
    'generated_at'        : str(datetime.now()),
}

out_path = LOGS_DIR / 'p13_reduced_transfer.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved to {out_path} ✓')
print('P13 complete ✓')

COMPARISON — Table 7 (Cross-Dataset Transfer, STATS19 test year 2022)
Configuration                                           AUROC    AUPRC   ECE(cal)    Brier
--------------------------------------------------------------------------------
  XGBoost — Original zero-shot (22 features, 12 mean-filled)   0.5182   0.2562     0.0088   0.1843
  XGBoost — Reduced zero-shot (10 common features)     0.5216   0.2586     0.0088   0.1842  <- NEW

  Delta (reduced vs original zero-shot):
    AUROC: +0.0034
    AUPRC: +0.0024

  No-skill AUPRC baseline (STATS19): 0.2438

Saved to /content/drive/MyDrive/SAE-XCrash/logs/wp7/p13_reduced_transfer.json ✓
P13 complete ✓
